<a href="https://colab.research.google.com/github/Jain-Dixon/MACHINE-LEARNING-WORKS/blob/main/assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)


In [ ]:
df=pd.read_csv('titanic_dataset.csv')
print(df.shape)

(891, 12)


In [ ]:
df.set_index('PassengerId', inplace=True)
print("\n" + "=" * 70)
print("STEP 2: PassengerId SET AS INDEX")
print("=" * 70)
print(df.head())


STEP 2: PassengerId SET AS INDEX
             Survived  Pclass                                               Name     Sex   Age  SibSp  Parch  \
PassengerId                                                                                                    
1                   0       3                            Braund, Mr. Owen Harris    male  22.0      1      0   
2                   1       1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1      0   
3                   1       3                             Heikkinen, Miss. Laina  female  26.0      0      0   
4                   1       1       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1      0   
5                   0       3                           Allen, Mr. William Henry    male  35.0      0      0   

                       Ticket     Fare Cabin Embarked  
PassengerId                                            
1                   A/5 21171   7.2500   NaN        S  
2            

In [ ]:
print(df.isnull().sum())

Survived      0
Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64


In [ ]:
def iqr_outlier_bounds(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper


outlier_cols = ['Age', 'Fare', 'SibSp']
fig, axes = plt.subplots(2, len(outlier_cols), figsize=(15, 8))
for i, col in enumerate(outlier_cols):
    lower, upper = iqr_outlier_bounds(df[col])
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"\n{col}: IQR bounds = [{lower:.2f}, {upper:.2f}] -> "
          f"{n_outliers} outliers found ({n_outliers/len(df)*100:.1f}%)")\

    sns.boxplot(y=df[col], ax=axes[0, i], color='salmon')
    axes[0, i].set_title(f'{col} - Before capping')


    df[col] = np.where(df[col] < lower, lower,
                        np.where(df[col] > upper, upper, df[col]))

    sns.boxplot(y=df[col], ax=axes[1, i], color='lightgreen')
    axes[1, i].set_title(f'{col} - After capping')

plt.tight_layout()
plt.savefig('outliers_before_after.png', dpi=100)
plt.close()
print("\nSaved boxplot comparison -> outliers_before_after.png")



Age: IQR bounds = [-6.69, 64.81] -> 11 outliers found (1.2%)

Fare: IQR bounds = [-26.72, 65.63] -> 116 outliers found (13.0%)

SibSp: IQR bounds = [-1.50, 2.50] -> 46 outliers found (5.2%)

Saved boxplot comparison -> outliers_before_after.png


In [ ]:
import pandas as pd

df = pd.read_csv('titanic_dataset.csv')
df.set_index('PassengerId', inplace=True)

In [ ]:
cols = ['Age', 'Fare', 'SibSp']

for col in cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

    print(f"{col}: {((df[col] < lower) | (df[col] > upper)).sum()} outliers")
    df[col] = df[col].clip(lower, upper)

Age: 11 outliers
Fare: 116 outliers
SibSp: 46 outliers


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

df = pd.read_csv('titanic_dataset.csv')

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})


X = df[['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]
y = df['Survived']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, accuracy_score(y_test, model.predict(X_test)))

Logistic Regression 0.8100558659217877
Random Forest 0.8044692737430168
